# COMP5329 - Deep Learning
**Tutorial 3 — Foundations of Deep Neural Networks**  
**Semester 1, 2026**

**Objectives:**
- Understand how data is represented and loaded in a deep learning pipeline
- Implement weight initialisation: zeros, Xavier, and He
- Implement activation functions (Sigmoid, Tanh, ReLU) with analytical derivatives
- Build a reusable `HiddenLayer` (WX + b) and an `MLP` from scratch
- Implement cross-entropy and MSE loss with their gradients
- Implement backpropagation **manually** — no `autograd`, no `torch.optim`
- Implement SGD, SGD with Momentum, and Adam optimisers from scratch
- Train models and visualise decision boundaries on a 2D classification task

## 1. The Role of Data in Deep Learning

Deep learning models are **function approximators**. Given a dataset of (input, label) pairs, a model learns a mapping $f_\theta : X \rightarrow y$ by adjusting its parameters $\theta$ to minimise a loss function $\mathcal{L}(f_\theta(X), y)$.

Data plays three distinct roles in this process:

| Split | Purpose | Typical fraction |
|-------|---------|------------------|
| **Training** | Parameter updates (gradient descent) | 70 – 80 % |
| **Validation** | Hyperparameter tuning, early stopping | 10 – 15 % |
| **Test** | Final unbiased evaluation | 10 – 15 % |

PyTorch provides two abstractions for data handling:

- **`Dataset`** — defines *what* a single sample is (`__len__` + `__getitem__`). Subclassing `Dataset` keeps data logic separate from model code and makes it trivial to swap datasets without touching the training loop.
- **`DataLoader`** — wraps a `Dataset` and handles batching, shuffling, and (optionally) multi-process data loading. Mini-batch training with a `DataLoader` is standard practice because it balances gradient noise and computational efficiency.

We will use a simple **2D two-class point dataset** as our running example. Points are drawn from two Gaussian clusters, and our model must learn the decision boundary that separates them.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import math

%matplotlib inline

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

### 1.1 Generating a 2D Toy Dataset

We sample $N$ points from two 2-D Gaussian distributions with equal covariance ($\sigma = 0.5 I$) but different means:

$$\text{Class 0}: \mu_0 = (-1.5,\,-1.5), \quad \text{Class 1}: \mu_1 = (+1.5,\,+1.5)$$

The two clusters overlap slightly near the origin, making this a non-trivial but learnable classification problem. We represent the dataset as **PyTorch tensors** from the start so every downstream computation remains in the PyTorch ecosystem.

In [ ]:
# ── Dataset parameters ────────────────────────────────────────────
N_PER_CLASS = 200   # samples per class
STD         = 0.5   # isotropic standard deviation

# ── Generate 2-D Gaussian clusters ─────────────────────────────────
class_0 = torch.randn(N_PER_CLASS, 2) * STD + torch.tensor([-1.5, -1.5])
class_1 = torch.randn(N_PER_CLASS, 2) * STD + torch.tensor([ 1.5,  1.5])

# Concatenate features and create integer labels (0 or 1)
X = torch.cat([class_0, class_1], dim=0)          # (400, 2)  float32
y = torch.cat([
    torch.zeros(N_PER_CLASS, dtype=torch.long),   # class 0
    torch.ones( N_PER_CLASS, dtype=torch.long),   # class 1
])                                                 # (400,)   int64

# Shuffle to remove class ordering
perm = torch.randperm(len(X))
X, y = X[perm], y[perm]

print(f"X : shape={X.shape},  dtype={X.dtype}")
print(f"y : shape={y.shape},  dtype={y.dtype}")
print(f"Class 0: {(y == 0).sum().item()} samples | Class 1: {(y == 1).sum().item()} samples")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(class_0[:, 0], class_0[:, 1],
            c="steelblue", label="Class 0",
            alpha=0.7, edgecolors="k", linewidths=0.3, s=25)
plt.scatter(class_1[:, 0], class_1[:, 1],
            c="tomato", label="Class 1",
            alpha=0.7, edgecolors="k", linewidths=0.3, s=25)
plt.xlabel("Feature $x_1$")
plt.ylabel("Feature $x_2$")
plt.title("2D Two-Class Point Dataset")
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis("equal")
plt.tight_layout()
plt.show()

### 1.2 Custom `Dataset` and `DataLoader`

Rather than relying on `torch.utils.data`, we implement both abstractions from scratch so the mechanics are fully visible.

**`Dataset`** — an abstract base class that defines the two-method contract every concrete dataset must satisfy:
- `__len__`  — total number of samples
- `__getitem__` — one `(features, label)` pair at a given index

**`DataLoader`** — wraps any `Dataset` and turns it into an iterator of mini-batches. Each time the training loop calls `for X_b, y_b in loader:`, the DataLoader:

1. Generates a **random permutation** of all sample indices (if `shuffle=True`)
2. Slices that permutation into consecutive chunks of size `batch_size`
3. Calls `dataset[i]` for each index in the chunk and **stacks** the results into tensors of shape `(B, D)` and `(B,)`

This pattern is identical across the whole course — only `__getitem__` changes to match the data type (images in Week 5, sequences in Week 6).

In [ ]:
class Dataset:
    """Abstract base class for datasets.

    Every concrete dataset must subclass this and implement:
        __len__      — return the total number of samples
        __getitem__  — return one (features, label) pair for index ``idx``
    """

    def __len__(self):
        raise NotImplementedError("Subclasses must implement __len__")

    def __getitem__(self, idx):
        raise NotImplementedError("Subclasses must implement __getitem__")


class DataLoader:
    """Mini-batch iterator — implemented without torch.utils.data.

    Algorithm
    ---------
    Each call to ``__iter__`` (i.e. each epoch) does three things:

    1. Decide visiting order
          shuffle=True  → torch.randperm(N)   (random permutation)
          shuffle=False → torch.arange(N)      (sequential)

    2. Slice the index sequence into consecutive windows of ``batch_size``

    3. For each window, call dataset[i] for every index i,
       then torch.stack the results into a (B, ...) batch tensor.

    Args:
        dataset    : any object implementing __len__ and __getitem__
        batch_size : samples per mini-batch (default 32)
        shuffle    : permute indices each epoch  (default True)
    """

    def __init__(self, dataset, batch_size: int = 32, shuffle: bool = True):
        self.dataset    = dataset
        self.batch_size = batch_size
        self.shuffle    = shuffle

    def __len__(self) -> int:
        """Number of mini-batches per epoch (ceiling division)."""
        return math.ceil(len(self.dataset) / self.batch_size)

    def __iter__(self):
        """Yield (X_batch, y_batch) pairs one mini-batch at a time."""
        n = len(self.dataset)

        # Step 1: index order for this epoch
        indices = torch.randperm(n) if self.shuffle else torch.arange(n)

        # Step 2 & 3: chunk indices, collect samples, stack into tensors
        for start in range(0, n, self.batch_size):
            batch_idx = indices[start : start + self.batch_size]

            samples = [self.dataset[i.item()] for i in batch_idx]
            X_batch = torch.stack([s[0] for s in samples])  # (B, D)
            y_batch = torch.stack([s[1] for s in samples])  # (B,)
            yield X_batch, y_batch

In [ ]:
class PointDataset(Dataset):
    """Generic (features, labels) dataset — subclasses our own Dataset.

    __getitem__ returns one (x_i, y_i) pair; our DataLoader stacks
    these into (B, D) and (B,) tensors for the training loop.

    Args:
        X : float tensor of shape (N, D)
        y : long  tensor of shape (N,)
    """

    def __init__(self, X: torch.Tensor, y: torch.Tensor):
        assert len(X) == len(y), "X and y must have the same number of samples."
        self.X = X
        self.y = y

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
BATCH_SIZE = 32

dataset = PointDataset(X, y)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Total samples  : {len(dataset)}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Batches/epoch  : {len(loader)}")

# Inspect one mini-batch
X_batch, y_batch = next(iter(loader))
print(f"\nSample batch   : X={X_batch.shape},  y={y_batch.shape}")
print(f"Label counts   : class-0={( y_batch==0).sum()}, class-1={(y_batch==1).sum()}")

---
## 2. Building Blocks of a Neural Network

A neural network is a composition of simple differentiable functions. Before assembling a full MLP, we implement each building block independently:

1. **Weight initialisation** — sets the starting point for optimisation
2. **Activation function** — introduces non-linearity and its own backward rule
3. **`HiddenLayer`** — a linear projection followed by an activation

Each component is implemented as a standalone function or class so that it can be **reused in later tutorials** (CNNs, RNNs) without modification.

### 2.1 Weight Initialisation

**Why does initialisation matter?**  
If all weights start at **zero**, every neuron in a layer receives exactly the same gradient and updates identically. The layer collapses to a single effective neuron regardless of its width — this is the **symmetry problem**.

Two principled strategies break symmetry while preventing vanishing / exploding gradients:

| Method | Formula | Derivation intuition | Best for |
|--------|---------|---------------------|----------|
| **Zeros** | $W = 0$ | — | ⚠ Never use |
| **Xavier** | $\sigma = \sqrt{\dfrac{2}{n_{\text{in}}+n_{\text{out}}}}$ | Keeps activation variance constant across layers (Glorot & Bengio, 2010) | sigmoid / tanh |
| **He** | $\sigma = \sqrt{\dfrac{2}{n_{\text{in}}}}$ | Compensates for ReLU zeroing out half the neurons (He et al., 2015) | ReLU |

Weights are drawn from $W \sim \mathcal{N}(0, \sigma^2)$ using the chosen $\sigma$.

In [ ]:
def init_weights(method: str, shape: tuple) -> torch.Tensor:
    """Return an initialised weight tensor.

    Args:
        method : "zeros" | "xavier" | "he"
        shape  : (fan_in, fan_out)

    Returns:
        torch.Tensor of shape ``shape`` with requires_grad=False.
    """
    fan_in, fan_out = shape

    if method == "zeros":
        # ⚠ All neurons receive identical gradients → symmetry problem.
        return torch.zeros(*shape)

    elif method == "xavier":
        # Glorot & Bengio (2010): balances variance across layers
        # for sigmoid / tanh activations.
        std = math.sqrt(2.0 / (fan_in + fan_out))
        return torch.randn(*shape) * std

    elif method == "he":
        # He et al. (2015): doubles Xavier variance to compensate for
        # ReLU zeroing out ~50 % of activations.
        std = math.sqrt(2.0 / fan_in)
        return torch.randn(*shape) * std

    else:
        raise ValueError(
            f"Unknown initialisation: '{method}'. "
            f"Choose from 'zeros', 'xavier', 'he'.")

In [ ]:
# Compare weight distributions for a layer with fan_in=256, fan_out=128
FAN_IN, FAN_OUT = 256, 128
SHAPE = (FAN_IN, FAN_OUT)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, method, color in zip(axes,
                              ["zeros", "xavier", "he"],
                              ["grey",  "steelblue", "tomato"]):
    W = init_weights(method, SHAPE)
    ax.hist(W.numpy().ravel(), bins=60, color=color, edgecolor="none", alpha=0.8)
    ax.set_title(f"{method.capitalize()}  (std={W.std():.4f})")
    ax.set_xlabel("Weight value")
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_ylabel("Count")
fig.suptitle(f"Weight distributions  (fan_in={FAN_IN}, fan_out={FAN_OUT})",
             fontsize=13)
plt.tight_layout()
plt.show()

### 2.2 Activation Functions

A **linear** layer alone is still a linear function — no matter how many you stack, the composition is linear and can only represent hyperplane decision boundaries. Activation functions introduce **non-linearity**, enabling the network to approximate any continuous function (Universal Approximation Theorem).

We implement three common activations with their exact **analytical derivatives**, which are required for backpropagation.

| Activation | Forward $f(z)$ | Derivative $f'(z)$ | Range |
|-----------|---------------|-------------------|-------|
| **Sigmoid** | $\dfrac{1}{1+e^{-z}}$ | $f(z)\cdot(1-f(z))$ | $(0,\,1)$ |
| **Tanh** | $\tanh(z)$ | $1 - \tanh^2(z)$ | $(-1,\,1)$ |
| **ReLU** | $\max(0, z)$ | $\mathbb{1}[z > 0]$ | $[0,\,\infty)$ |

The `Activation` class caches the **pre-activation** $z$ during the forward pass so that the backward pass can compute $f'(z)$ without recomputing $z$.

In [ ]:
class Activation:
    """Stateful activation function with forward / backward interface.

    Each HiddenLayer owns one Activation instance. The instance caches
    the pre-activation tensor ``z`` during the forward pass so that the
    backward pass can compute the element-wise derivative without
    recomputing ``z`` from scratch.

    Supported names: "sigmoid", "tanh", "relu", "linear" (or None).
    """

    def __init__(self, name: str = "relu"):
        self.name    = name
        self._z      = None   # pre-activation cache

    # ── Forward: apply f(z) and cache z for backward ────────────────
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        self._z = z
        if self.name == "sigmoid":
            return 1.0 / (1.0 + torch.exp(-z))
        elif self.name == "tanh":
            return torch.tanh(z)
        elif self.name == "relu":
            return z.clamp(min=0)
        else:                         # "linear" or None
            return z

    # ── Backward: multiply upstream gradient by f'(z) ───────────────
    def backward(self, grad: torch.Tensor) -> torch.Tensor:
        """grad : dL/d(activation)  →  returns dL/dz = grad · f'(z)"""
        z = self._z
        if self.name == "sigmoid":
            s = 1.0 / (1.0 + torch.exp(-z))
            return grad * s * (1.0 - s)
        elif self.name == "tanh":
            return grad * (1.0 - torch.tanh(z) ** 2)
        elif self.name == "relu":
            return grad * (z > 0).float()
        else:
            return grad

In [ ]:
z = torch.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
config = [
    ("sigmoid", "steelblue"),
    ("tanh",    "darkorange"),
    ("relu",    "seagreen"),
]

for ax, (name, color) in zip(axes, config):
    act   = Activation(name)
    fwd   = act.forward(z)
    # gradient of output w.r.t. z: pass ones as upstream gradient
    deriv = act.backward(torch.ones_like(z))
    ax.plot(z.numpy(), fwd.numpy(),   color=color,  lw=2,   label=f"f(z)")
    ax.plot(z.numpy(), deriv.numpy(), color=color,  lw=2,
            linestyle="--", alpha=0.7, label="f'(z)")
    ax.axhline(0, color="black", lw=0.6)
    ax.axvline(0, color="black", lw=0.6)
    ax.set_title(name.capitalize(), fontsize=13)
    ax.set_xlabel("z")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("Activation Functions and Their Derivatives", fontsize=13)
plt.tight_layout()
plt.show()

### 2.3 The Hidden Layer: Linear Transformation

Each layer computes:

$$\text{output} = f\!\left(\underbrace{X \cdot W}_{\text{projection}} + b\right)$$

| Symbol | Shape | Description |
|--------|-------|-------------|
| $X$ | $(B,\, n_{\text{in}})$ | input batch ($B$ = batch size) |
| $W$ | $(n_{\text{in}},\, n_{\text{out}})$ | weight matrix |
| $b$ | $(1,\, n_{\text{out}})$ | bias (broadcast across batch) |
| output | $(B,\, n_{\text{out}})$ | activated features |

**Backward pass derivation (chain rule)**

Let $\delta_{\text{out}}$ be the upstream gradient $\partial\mathcal{L}/\partial\text{output}$, and let $z = XW + b$ be the pre-activation. Then:

$$\delta_z = f'(z) \odot \delta_{\text{out}} \qquad \text{(element-wise, via Activation.backward)}$$

$$\nabla_W = X^\top \delta_z, \quad \nabla_b = \sum_{\text{batch}} \delta_z, \quad \delta_{\text{in}} = \delta_z W^\top$$

$\delta_{\text{in}}$ is passed to the **previous** layer to continue backpropagation.

> **Design note:** `HiddenLayer` accepts `activation` and `initialization` as constructor arguments so that the same class can be used for the FC heads in CNN and RNN models in later weeks — only the choice of activation and init strategy changes.

In [ ]:
class HiddenLayer:
    """Fully-connected layer: output = activation(X @ W + b).

    This is the fundamental building block shared by MLP (Week 3),
    CNN FC heads (Week 5), and RNN output layers (Week 6).

    Args:
        n_in           : number of input features
        n_out          : number of output neurons
        activation     : activation function name (default "relu")
        initialization : weight init strategy   (default "xavier")

    Example:
        h = HiddenLayer(2, 16, activation="relu", initialization="he")
        out = h.forward(X_batch)           # forward
        dx, dW, db = h.backward(grad_out)  # manual backprop
    """

    def __init__(self, n_in: int, n_out: int,
                 activation: str = "relu",
                 initialization: str = "xavier"):
        # Parameters
        self.W   = init_weights(initialization, (n_in, n_out))  # (n_in, n_out)
        self.b   = torch.zeros(1, n_out)                         # (1,    n_out)
        # Activation sub-module
        self.act = Activation(activation)
        # Cache: input X stored here during forward for use in backward
        self._x  = None

    # ── Forward pass ─────────────────────────────────────────────────
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, n_in)  →  output : (B, n_out)"""
        self._x = x                       # cache input
        z       = x @ self.W + self.b     # linear transform  (B, n_out)
        return self.act.forward(z)         # non-linearity     (B, n_out)

    # ── Backward pass (chain rule) ────────────────────────────────────
    def backward(self, grad: torch.Tensor):
        """Backpropagate gradient through this layer.

        Args:
            grad : dL/d(output), shape (B, n_out)

        Returns:
            dx : dL/dX,  shape (B, n_in)    — passed to previous layer
            dW : dL/dW,  shape (n_in, n_out) — used by the optimiser
            db : dL/db,  shape (1,    n_out) — used by the optimiser
        """
        dz = self.act.backward(grad)            # (B, n_out)  through activation
        x  = self._x                            # (B, n_in)   cached input
        dW = x.T  @ dz                          # (n_in, n_out)
        db = dz.sum(dim=0, keepdim=True)        # (1,    n_out)
        dx = dz   @ self.W.T                    # (B,    n_in)
        return dx, dW, db

---
## 3. Loss Functions

The loss function quantifies how wrong the model is. Its gradient w.r.t. the model's output is the "error signal" that initiates backpropagation.

### 3.1 Softmax
For a $C$-class problem the model outputs $C$ raw scores (logits) $z_1, \ldots, z_C$. **Softmax** converts them into a probability distribution:

$$p_k = \frac{e^{z_k}}{\sum_{j=1}^{C} e^{z_j}}$$

Numerically stable version (subtract $z_{\max}$ before exp to avoid overflow):

$$p_k = \frac{e^{z_k - z_{\max}}}{\sum_j e^{z_j - z_{\max}}}$$

### 3.2 Cross-Entropy Loss
Penalises low confidence in the correct class:

$$\mathcal{L}_{\text{CE}} = -\frac{1}{N} \sum_{i=1}^{N} \log p_{i,\,y_i}$$

Combined gradient of (softmax + cross-entropy) w.r.t. the logits has an elegant form:

$$\frac{\partial \mathcal{L}}{\partial z_k} = \frac{p_k - \mathbb{1}[k = y]}{N}$$

We simply subtract 1 from the predicted probability of the **true class** and divide by $N$.

### 3.3 MSE Loss (Euclidean Distance)
Used for regression or single-output problems:

$$\mathcal{L}_{\text{MSE}} = \frac{1}{N}\|\hat{y} - y\|^2, \qquad \frac{\partial \mathcal{L}}{\partial \hat{y}} = \frac{2}{N}(\hat{y} - y)$$

In [ ]:
# ── softmax stays a standalone helper (no state) ───────────────────
def softmax(logits: torch.Tensor) -> torch.Tensor:
    """Numerically stable softmax along the class dimension."""
    shifted = logits - logits.max(dim=1, keepdim=True).values
    e = torch.exp(shifted)
    return e / e.sum(dim=1, keepdim=True)


# ── Loss base class ─────────────────────────────────────────────────
class Loss:
    """Abstract base class for loss functions.

    Mirrors the forward / backward interface of Activation and
    HiddenLayer so that every differentiable component in the
    network follows the same design pattern:

        value = loss.forward(predictions, targets)
        grad  = loss.backward()

    forward() caches whatever the backward pass will need, so the
    caller never has to pass intermediate tensors twice.
    """

    def forward(self, *args):
        raise NotImplementedError

    def backward(self):
        raise NotImplementedError


# ── Cross-Entropy Loss ───────────────────────────────────────────────
class CrossEntropyLoss(Loss):
    """Softmax cross-entropy loss.

    forward() accepts raw logits (not probabilities), applies softmax
    internally, and caches the probabilities for backward().

    Combined gradient of (softmax + CE) w.r.t. the input logits:
        dL/dz_k = (p_k - 1{k == y}) / N
    """

    def __init__(self):
        self._probs  = None   # cached softmax output
        self._labels = None   # cached true class indices

    def forward(self, logits: torch.Tensor,
                labels: torch.Tensor) -> torch.Tensor:
        """Compute loss and cache (probs, labels) for backward.

        Args:
            logits : (N, C) raw scores
            labels : (N,)  integer class indices
        Returns:
            scalar loss tensor
        """
        self._probs  = softmax(logits)         # (N, C)  — cached
        self._labels = labels                  # (N,)    — cached
        N      = labels.shape[0]
        p_true = self._probs[torch.arange(N), labels]   # (N,)
        return -torch.log(p_true + 1e-9).mean()

    def backward(self) -> torch.Tensor:
        """Gradient of (softmax + CE) w.r.t. the input logits.

        Returns:
            grad : (N, C) tensor  — passed into MLP.backward()
        """
        assert self._probs is not None, "Call forward() before backward()"
        N    = self._labels.shape[0]
        grad = self._probs.clone()
        grad[torch.arange(N), self._labels] -= 1.0   # p_k - 1 at true class
        return grad / N


# ── MSE Loss ─────────────────────────────────────────────────────────
class MSELoss(Loss):
    """Mean squared error: L = (1/N) ||pred - target||^2.

    Used for regression or single-output problems.
    """

    def __init__(self):
        self._pred   = None
        self._target = None

    def forward(self, pred: torch.Tensor,
                target: torch.Tensor) -> torch.Tensor:
        self._pred   = pred
        self._target = target
        return ((pred - target) ** 2).mean()

    def backward(self) -> torch.Tensor:
        """Gradient of MSE w.r.t. pred: (2/N)(pred - target)"""
        N = self._pred.shape[0]
        return 2.0 * (self._pred - self._target) / N


# ── Sanity check ────────────────────────────────────────────────────
ce = CrossEntropyLoss()
logits_test = torch.tensor([[2.0, 1.0, 0.5], [0.5, 2.5, 1.0]])
labels_test = torch.tensor([0, 1])

loss_val = ce.forward(logits_test, labels_test)
grad_val = ce.backward()

print("Softmax probs (rows sum to 1):")
print(ce._probs.round(decimals=4))
print(f"CE loss : {loss_val.item():.4f}")
print(f"\nCE gradient w.r.t. logits:")
print(grad_val.round(decimals=4))

---
## 4. Backpropagation

**Backpropagation** is the application of the chain rule to compute $\partial\mathcal{L}/\partial\theta$ for every parameter $\theta$ in a neural network, traversing layers in reverse order.

For a network with $L$ layers and a batch input $X$:

$$\text{Forward:} \quad h^{(0)} = X, \quad h^{(l)} = f_l(h^{(l-1)} W^{(l)} + b^{(l)})$$

$$\text{Backward:} \quad \delta^{(L)} = \nabla_{h^{(L)}}\mathcal{L}, \quad \delta^{(l-1)} = \delta^{(l)} {W^{(l)}}^\top \odot f_l'\!\left(z^{(l-1)}\right)$$

$$\nabla_{W^{(l)}} = {h^{(l-1)}}^\top \delta^{(l)}, \quad \nabla_{b^{(l)}} = \sum_{\text{batch}} \delta^{(l)}$$

In code, `HiddenLayer.backward(grad)` implements exactly these equations for a single layer. The `MLP` class chains them in **reverse order** and collects parameter gradients.

> **Note:** We do not call `.backward()` or use `torch.no_grad()`. The tensors here have `requires_grad=False` (the default). All gradient computation is **explicit and manual**.

In [ ]:
class MLP:
    """Multi-layer perceptron assembled from HiddenLayer objects.

    Args:
        layer_dims     : list of ints — [input_dim, h1, h2, ..., output_dim]
        activations    : list of str  — one activation name per layer transition
        initialization : weight init strategy shared by all layers

    Example:
        model = MLP([2, 16, 8, 2],
                    activations=["relu", "relu", "linear"],
                    initialization="xavier")
    """

    def __init__(self, layer_dims: list, activations: list,
                 initialization: str = "xavier"):
        assert len(activations) == len(layer_dims) - 1, (
            "Need exactly one activation per layer transition.")
        self.layers = [
            HiddenLayer(layer_dims[i], layer_dims[i + 1],
                        activation=activations[i],
                        initialization=initialization)
            for i in range(len(layer_dims) - 1)
        ]

    # ── Forward: chain layers left to right ─────────────────────────
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer.forward(x)
        return x   # raw logits (no softmax — kept separate for flexibility)

    # ── Backward: traverse layers right to left ─────────────────────
    def backward(self, grad: torch.Tensor) -> list:
        """Backpropagate loss gradient through all layers.

        Args:
            grad : dL/d(output), shape (B, output_dim)

        Returns:
            List of (dW, db) tuples in **forward order** (layer 0 first).
        """
        param_grads = []
        for layer in reversed(self.layers):
            grad, dW, db = layer.backward(grad)
            param_grads.append((dW, db))
        return list(reversed(param_grads))   # restore forward order

### 4.1 Training with Vanilla SGD

**Stochastic Gradient Descent (SGD)** is the simplest optimiser. After computing the loss on a mini-batch, it updates every parameter by a small step in the **negative gradient direction**:

$$\theta \leftarrow \theta - \eta \cdot \nabla_\theta \mathcal{L}$$

where $\eta$ (the **learning rate**) controls step size. A learning rate that is too large causes divergence; too small causes slow convergence.

The training loop:
1. Sample a mini-batch $(X_b, y_b)$ from the `DataLoader`
2. **Forward**: compute logits → softmax → loss
3. **Backward**: compute $\partial\mathcal{L}/\partial z_{\text{out}}$ → call `MLP.backward()` → collect $(\nabla W, \nabla b)$ per layer
4. **Update**: subtract $\eta \cdot \nabla$ from each parameter

In [ ]:
class SGD:
    """Vanilla stochastic gradient descent.

    Update rule:  theta <- theta - lr * grad

    Args:
        lr : learning rate (step size)
    """

    def __init__(self, lr: float = 0.01):
        self.lr = lr

    def step(self, layers: list, grads: list):
        """Update all layer parameters in-place.

        Args:
            layers : list of HiddenLayer objects
            grads  : list of (dW, db) in same order as layers
        """
        for layer, (dW, db) in zip(layers, grads):
            layer.W -= self.lr * dW
            layer.b -= self.lr * db

In [ ]:
def train(model, criterion, optimizer, X, y,
          n_epochs=500, batch_size=32, verbose=True):
    """Generic training loop for any (model, criterion, optimizer) combination.

    Args:
        model      : MLP instance
        criterion  : Loss instance (e.g. CrossEntropyLoss())
        optimizer  : SGD | SGDMomentum | Adam instance
        X          : (N, D) float tensor of features
        y          : (N,)   long  tensor of integer labels
        n_epochs   : number of full passes through the dataset
        batch_size : mini-batch size
        verbose    : print loss every 100 epochs

    Returns:
        List of per-epoch average losses.
    """
    loader = DataLoader(PointDataset(X, y),
                        batch_size=batch_size, shuffle=True)
    losses = []

    for epoch in range(1, n_epochs + 1):
        epoch_loss = 0.0

        for X_b, y_b in loader:
            # ── Forward ──────────────────────────────────────────────
            logits = model.forward(X_b)            # (B, n_classes)
            loss   = criterion.forward(logits, y_b) # scalar
            epoch_loss += loss.item()

            # ── Backward ─────────────────────────────────────────────
            grad_out = criterion.backward()          # (B, n_classes)
            grads    = model.backward(grad_out)      # [(dW, db), ...]

            # ── Parameter update ─────────────────────────────────────
            optimizer.step(model.layers, grads)

        avg = epoch_loss / len(loader)
        losses.append(avg)
        if verbose and epoch % 100 == 0:
            print(f"Epoch {epoch:4d}/{n_epochs}  |  Loss: {avg:.4f}")

    return losses

In [ ]:
# ── Helper: create a fresh model with the same seed ────────────────
def make_model(seed=0):
    torch.manual_seed(seed)
    return MLP(
        layer_dims=[2, 32, 16, 2],
        activations=["relu", "relu", "linear"],
        initialization="he"
    )

# ── Train with vanilla SGD ───────────────────────────────────────────
model_sgd = make_model()
opt_sgd   = SGD(lr=0.05)
crit_sgd  = CrossEntropyLoss()

print("Training with vanilla SGD...")
losses_sgd = train(model_sgd, crit_sgd, opt_sgd, X, y, n_epochs=500, batch_size=32)

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary", ax=None):
    """Render model decision surface over the 2D feature space."""
    x_min = X[:, 0].min().item() - 0.5
    x_max = X[:, 0].max().item() + 0.5
    y_min = X[:, 1].min().item() - 0.5
    y_max = X[:, 1].max().item() + 0.5
    step  = 0.03

    xx, yy = np.meshgrid(np.arange(x_min, x_max, step),
                          np.arange(y_min, y_max, step))
    grid   = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    logits = model.forward(grid)
    Z      = softmax(logits)[:, 1].detach().numpy().reshape(xx.shape)

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    cf = ax.contourf(xx, yy, Z, levels=50, cmap="RdBu", alpha=0.8)
    plt.colorbar(cf, ax=ax, label="P(Class 1)")
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="steelblue", edgecolors="k",
               linewidths=0.4, s=18, label="Class 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="tomato", edgecolors="k",
               linewidths=0.4, s=18, label="Class 1")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
    ax.legend(fontsize=8)
    return ax


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(losses_sgd, color="steelblue", lw=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("SGD — Training Loss")
axes[0].grid(True, alpha=0.3)

# Decision boundary
plot_decision_boundary(model_sgd, X, y, title="SGD — Decision Boundary", ax=axes[1])

plt.tight_layout()
plt.show()

---
## 5. Better Optimisers

Vanilla SGD has two main limitations:

1. **Oscillation** — in loss landscapes with narrow ravines, gradients point mostly across the ravine rather than towards the minimum, causing slow zig-zagging.
2. **Uniform step size** — all parameters use the same learning rate, but some parameters may receive consistently small or noisy gradients.

Two families of improvements address these:
- **Momentum-based** — smooth the update direction by accumulating past gradients
- **Adaptive learning rate** — rescale each parameter's step by its gradient history

We implement both from scratch with no `torch.optim` calls.

### 5.1 SGD with Momentum

Momentum accumulates a **velocity** vector in the gradient direction, analogous to a ball rolling down a hill that gains speed in consistent directions and loses speed when the gradient oscillates.

$$v_t = \mu \cdot v_{t-1} \;-\; \eta \cdot \nabla_\theta \mathcal{L}$$

$$\theta_t = \theta_{t-1} + v_t$$

| Hyperparameter | Typical value | Effect |
|---------------|--------------|--------|
| $\eta$ (learning rate) | 0.01 | step size |
| $\mu$ (momentum) | 0.9 | how much past velocity is retained |

With $\mu = 0.9$, the effective step is roughly $\frac{\eta}{1 - \mu} = 10\times$ the vanilla SGD step in a direction of persistent gradient. The velocity is initialised to zero and updated lazily on first call.

In [ ]:
class SGDMomentum:
    """SGD with classical momentum.

    Update rule:
        v  <- mu * v  - lr * grad
        theta <- theta + v

    Velocity tensors are initialised lazily on the first call to ``step``
    so that layer shapes do not need to be specified at construction time.

    Args:
        lr       : learning rate
        momentum : momentum coefficient (default 0.9)
    """

    def __init__(self, lr: float = 0.01, momentum: float = 0.9):
        self.lr  = lr
        self.mu  = momentum
        self._vW = None   # lazily initialised velocity for W
        self._vb = None   # lazily initialised velocity for b

    def step(self, layers: list, grads: list):
        # Lazy initialisation on first call
        if self._vW is None:
            self._vW = [torch.zeros_like(l.W) for l in layers]
            self._vb = [torch.zeros_like(l.b) for l in layers]

        for i, (layer, (dW, db)) in enumerate(zip(layers, grads)):
            # Accumulate velocity
            self._vW[i] = self.mu * self._vW[i] - self.lr * dW
            self._vb[i] = self.mu * self._vb[i] - self.lr * db
            # Apply velocity
            layer.W += self._vW[i]
            layer.b += self._vb[i]

### 5.2 Adam (Adaptive Moment Estimation)

Adam (Kingma & Ba, 2014) combines momentum with **per-parameter adaptive learning rates**. It maintains two running averages for each parameter:

| Moment | Update | Meaning |
|--------|--------|---------|
| 1st (mean) | $m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t$ | EMA of gradients |
| 2nd (var.) | $v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$ | EMA of squared gradients |

Both estimates are **biased towards zero** early in training (because $m_0 = v_0 = 0$). Bias correction divides by $(1 - \beta^t)$:

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

Final parameter update:

$$\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon}$$

Parameters with **large, noisy gradients** get a small effective step (large $\hat{v}_t$ in the denominator). Parameters with **small, consistent gradients** get a larger relative step. Typical defaults: $\eta=0.001$, $\beta_1=0.9$, $\beta_2=0.999$, $\varepsilon=10^{-8}$.

In [ ]:
class Adam:
    """Adaptive Moment Estimation (Kingma & Ba, 2014).

    Update rule:
        m  <- beta1 * m  + (1 - beta1) * g          # 1st moment
        v  <- beta2 * v  + (1 - beta2) * g^2        # 2nd moment
        m_hat = m / (1 - beta1^t)                   # bias correction
        v_hat = v / (1 - beta2^t)
        theta <- theta - lr * m_hat / (sqrt(v_hat) + eps)

    All moment tensors are initialised lazily on the first ``step`` call.

    Args:
        lr    : learning rate (default 1e-3)
        beta1 : 1st-moment decay (default 0.9)
        beta2 : 2nd-moment decay (default 0.999)
        eps   : numerical stability constant (default 1e-8)
    """

    def __init__(self, lr: float = 1e-3,
                 beta1: float = 0.9, beta2: float = 0.999,
                 eps: float = 1e-8):
        self.lr    = lr
        self.b1    = beta1
        self.b2    = beta2
        self.eps   = eps
        self.t     = 0
        self._mW   = self._mb = self._vW = self._vb = None  # lazy init

    def step(self, layers: list, grads: list):
        if self._mW is None:   # initialise on first call
            self._mW = [torch.zeros_like(l.W) for l in layers]
            self._mb = [torch.zeros_like(l.b) for l in layers]
            self._vW = [torch.zeros_like(l.W) for l in layers]
            self._vb = [torch.zeros_like(l.b) for l in layers]

        self.t += 1

        for i, (layer, (dW, db)) in enumerate(zip(layers, grads)):
            # ── 1st moment (mean) ────────────────────────────────────
            self._mW[i] = self.b1 * self._mW[i] + (1 - self.b1) * dW
            self._mb[i] = self.b1 * self._mb[i] + (1 - self.b1) * db
            # ── 2nd moment (uncentred variance) ──────────────────────
            self._vW[i] = self.b2 * self._vW[i] + (1 - self.b2) * dW ** 2
            self._vb[i] = self.b2 * self._vb[i] + (1 - self.b2) * db ** 2
            # ── Bias correction ───────────────────────────────────────
            mW_hat = self._mW[i] / (1 - self.b1 ** self.t)
            mb_hat = self._mb[i] / (1 - self.b1 ** self.t)
            vW_hat = self._vW[i] / (1 - self.b2 ** self.t)
            vb_hat = self._vb[i] / (1 - self.b2 ** self.t)
            # ── Parameter update ──────────────────────────────────────
            layer.W -= self.lr * mW_hat / (torch.sqrt(vW_hat) + self.eps)
            layer.b -= self.lr * mb_hat / (torch.sqrt(vb_hat) + self.eps)

### 5.3 Comparing Optimisers

We train three identical models (same architecture, same weight initialisation) using three different optimisers and plot their learning curves side by side.

To ensure a fair comparison:
- The random seed is reset before each model is constructed → identical starting weights.
- The same dataset and batch size are used for all three runs.
- Learning rates are tuned individually to representative defaults for each optimiser.

In [ ]:
N_EPOCHS = 500

configs = [
    ("SGD",          make_model(), CrossEntropyLoss(), SGD(lr=0.05),              "steelblue"),
    ("SGD+Momentum", make_model(), CrossEntropyLoss(), SGDMomentum(lr=0.01, momentum=0.9), "darkorange"),
    ("Adam",         make_model(), CrossEntropyLoss(), Adam(lr=1e-3),              "seagreen"),
]

all_losses = {}
for name, model, crit, opt, _ in configs:
    print(f"\n── Training with {name} ──")
    all_losses[name] = train(model, crit, opt, X, y,
                             n_epochs=N_EPOCHS, batch_size=32)

# ── Loss curves ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
for (name, _, _, _, color) in configs:
    ax.plot(all_losses[name], label=name, color=color, lw=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Optimiser Comparison — Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Decision boundaries for Adam (best performer) ───────────────────
adam_model = configs[2][1]   # model is always at index 1
plot_decision_boundary(adam_model, X, y,
                       title="Adam — Decision Boundary", ax=axes[1])

plt.tight_layout()
plt.show()

# ── Final loss summary ───────────────────────────────────────────────
print("\nFinal loss after training:")
for name, _, _, _ in configs:
    print(f"  {name:<20s}: {all_losses[name][-1]:.4f}")

---
## 6. Claude Code — Command Reference

Week 2 introduced Claude Code as a tool. This section covers the **commands** you will use in everyday sessions: how to undo a bad edit, manage long conversations, switch models, control reasoning depth, and build your own reusable slash commands.

---

### 6.1 Basic Commands

#### Session & History Management

| Command | What it does |
|---------|--------------|
| `Esc` `Esc` &nbsp;or&nbsp; `/rewind` | Open a checkpoint menu — roll back conversation, code changes, or both to any earlier prompt in the session |
| `claude --continue` | From the terminal: resume the most recent session in the current directory |
| `claude --resume` | From the terminal: open an interactive session picker; navigate with `↑ ↓`, press `Enter` to load |
| `/resume` | Inside an active session: switch to a different saved conversation without leaving the terminal |
| `/compact [focus]` | Summarise the full conversation history into a compressed form, freeing up context window space |
| `/clear` | Wipe the current conversation transcript (the session can still be resumed later) |

**When to use each history command:**

| Situation | Best command |
|-----------|-------------|
| Claude made a bad edit and you want to undo it | `/rewind` → *Restore code and conversation* |
| You only want to undo Claude's last reply, keeping code as-is | `/rewind` → *Restore conversation* |
| Context window is nearly full mid-task | `/compact focus on recent code changes` |
| You want a completely fresh start on a new topic | `/clear` |
| You closed the terminal and want to pick up where you left off | `claude --resume` |

> **Rewind checkpoints** cover only file edits made by Claude's own tools. Changes from shell commands (e.g. `mv`, `rm`) and edits you made manually outside Claude Code are **not tracked**. For permanent history, use `git`.

---

#### Project Utility Commands

| Command | What it does |
|---------|--------------|
| `/commit` | Stage all modified tracked files and create a git commit; Claude generates the commit message automatically |
| `/model` | Open the model picker mid-session — switch between Haiku, Sonnet, and Opus without restarting |
| `/cost` | Display total token usage and estimated API cost for the current session |

**Choosing a model for the task:**

| Model | Strength | Typical use in this course |
|-------|----------|---------------------------|
| **Haiku** | Fast, cheap | Quick syntax questions, one-liner fixes |
| **Sonnet** | Balanced speed + quality | Writing training loops, debugging shape errors |
| **Opus** | Strongest reasoning | Deriving backprop equations, architectural decisions |

> **Note on `/cost`:** Relevant only for API-key accounts. Claude Pro / Max subscribers have usage bundled in their plan — use `/stats` for usage patterns instead.

In [ ]:
# All commands below are run in your TERMINAL inside an active Claude Code session.
# Do NOT run them here in the notebook.
#
# ── Session management ────────────────────────────────────────────────────────
#
#   Undo the last change (interactive menu):
#     Press Esc Esc  — or type:  /rewind
#
#   Resume the most recent session (from the terminal, before starting Claude):
#     claude --continue
#
#   Pick any past session from an interactive list:
#     claude --resume
#
#   Switch to a different session from inside an active session:
#     /resume
#
# ── Context management ────────────────────────────────────────────────────────
#
#   Summarise conversation to free up context (optional focus hint):
#     /compact
#     /compact focus on model architecture changes and training loop
#
#   Clear the transcript entirely and start fresh:
#     /clear
#
# ── Project utilities ─────────────────────────────────────────────────────────
#
#   Commit all changes with an AI-generated message:
#     /commit
#
#   Switch model mid-session (opens interactive picker):
#     /model
#
#   Switch to a specific model directly:
#     /model opus
#     /model sonnet
#     /model haiku
#
#   Show token usage and cost for this session:
#     /cost

---

### 6.2 Advanced Commands — Controlling Reasoning Depth

For straightforward tasks (fixing a typo, explaining a function) the default model response is adequate. For harder problems — deriving a backpropagation formula, diagnosing a subtle numerical instability, or designing an architecture — you can instruct Claude to spend more effort thinking before it responds.

#### Official method: effort level (Opus models)

When using an Opus model, the `/model` command exposes an **effort level slider**. Press `←` / `→` to move between `low`, `medium`, and `high`.

| Effort | Thinking tokens | Best for |
|--------|----------------|----------|
| `low` | Minimal | Quick lookups, simple edits |
| `medium` | Moderate (default) | Most coding tasks |
| `high` | Maximum | Hard reasoning, complex debugging, mathematical derivations |

You can also set effort before launching Claude:
```
CLAUDE_CODE_EFFORT_LEVEL=high claude
```

To view Claude's internal reasoning as it works, press `Ctrl+O` to toggle **verbose mode** — thinking steps appear as grey italic text.

To disable extended thinking entirely:
```
MAX_THINKING_TOKENS=0 claude
```

---

#### Prompt-level guidance

No official "magic keywords" allocate extra thinking tokens, but how you phrase the prompt shapes the quality of reasoning. These patterns are well-established in practice:

| Prompt phrasing | Effect | Example |
|----------------|--------|---------|
| `"Think step by step about..."` | Encourages an explicit chain of reasoning before writing code | *"Think step by step about why this gradient vanishes in deep sigmoid networks."* |
| `"First reason about X, then implement Y"` | Separates planning from execution, reducing early commitment to a flawed approach | *"First reason about the correct tensor shapes, then implement the forward pass."* |
| `"Think carefully before answering — check your maths"` | Useful for loss derivations or index arithmetic where off-by-one errors are common | *"Think carefully before answering — verify each matrix dimension in the backprop equations."* |

> **Recommended pattern for hard tasks:** set effort to `high` with `/model`, then use a structured prompt. The effort level ensures the thinking budget is available; the prompt ensures it is used productively.

---

### 6.3 Custom Slash Commands

Repetitive tasks — running a fixed evaluation suite, generating a standard experiment report, or reviewing a file against course style conventions — can be packaged as **custom slash commands**. Once created, a command appears in `/` autocomplete and can be invoked with optional arguments.

#### How it works

A custom command is a plain Markdown file. When you type `/your-command`, Claude reads the file and follows its instructions.

**File locations:**

| Scope | Path | Available in |
|-------|------|--------------|
| All your projects | `~/.claude/commands/<name>.md` | Every project on your machine |
| This project only | `.claude/commands/<name>.md` | Current repository only |

The filename (without `.md`) becomes the slash command name. Lowercase and hyphens only.

**File format:**

```
---
description: One-line summary shown in autocomplete
argument-hint: [optional description of expected argument]
---

Instructions for Claude, written in plain English.

Use $ARGUMENTS for the full argument string.
Use $0, $1, $2 … for individual space-separated arguments.
```

#### Example 1 — Evaluate a model on the toy dataset

Create `~/.claude/commands/eval-model.md`:

```
---
description: Run accuracy evaluation on the 2D toy dataset and report results
argument-hint: [variable name of the model to evaluate]
---

Evaluate the model stored in variable $ARGUMENTS on the toy dataset (X, y).

Steps:
1. Run model.forward(X) to get logits
2. Compute predicted class as argmax of logits
3. Compare predictions to y and compute accuracy
4. Print a clear summary: accuracy, number correct, total samples
5. Plot the decision boundary using plot_decision_boundary()

Do not modify the model or the dataset.
```

Usage: `/eval-model model_sgd`

---

#### Example 2 — Debug a shape mismatch

Create `.claude/commands/debug-shape.md` (project-level):

```
---
description: Diagnose a tensor shape mismatch error from a traceback
argument-hint: [paste the error message or describe where it occurs]
---

Debug the following tensor shape error: $ARGUMENTS

Steps:
1. Locate the line that raised the error
2. Print the shapes of all tensors involved at that line
3. Trace the shape back through each operation to find where the mismatch originates
4. Propose a minimal fix and explain why it is correct
```

Usage: `/debug-shape "RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x16 and 8x2)"`

---

> **Tip:** Add persistent context to your commands using `CLAUDE.md`. Any rules or conventions written there are automatically available to every custom command without you needing to repeat them in the command file.

In [ ]:
# ── Controlling reasoning depth ───────────────────────────────────────────────
# All commands are run in your TERMINAL — not in this notebook.
#
# Set effort level before launching (Opus models):
#   CLAUDE_CODE_EFFORT_LEVEL=high claude
#
# Or change effort mid-session:
#   /model         (then press ← → to move the effort slider)
#
# View Claude's thinking as it runs:
#   Ctrl+O         (toggles verbose mode — thinking shown as grey italic text)
#
# Disable extended thinking:
#   MAX_THINKING_TOKENS=0 claude
#
# ── Example: prompts that encourage structured reasoning ─────────────────────
#
# Simple question (no extra guidance needed):
#   "What does .detach() do in PyTorch?"
#
# Medium complexity — ask for step-by-step:
#   "Think step by step about why gradient flow breaks in a deep sigmoid network."
#
# High complexity — separate planning from implementation:
#   "First reason about the correct tensor shapes for the attention mechanism,
#    then implement scaled dot-product attention."
#
# Mathematical derivation — ask for careful checking:
#   "Derive the gradient of cross-entropy loss w.r.t. the softmax input.
#    Think carefully and verify each step."
#
# ── Creating custom slash commands ────────────────────────────────────────────
#
# Step 1: Create the commands directory (run once):
#   mkdir -p ~/.claude/commands                # personal (all projects)
#   mkdir -p .claude/commands                  # project-level (this repo only)
#
# Step 2: Write a Markdown file — e.g. ~/.claude/commands/eval-model.md
#
#   ---
#   description: Evaluate a model on the toy dataset and report accuracy
#   argument-hint: [model variable name]
#   ---
#
#   Evaluate the model in variable $ARGUMENTS on the toy dataset (X, y).
#   1. Compute logits via model.forward(X)
#   2. Take argmax to get predicted classes
#   3. Compute and print accuracy
#   4. Plot the decision boundary
#
# Step 3: Use it (the command appears in / autocomplete immediately):
#   /eval-model model_adam
#
# ── Argument placeholders ─────────────────────────────────────────────────────
#
#   $ARGUMENTS        — entire argument string after the command name
#   $0, $1, $2 …      — individual space-separated arguments
#
#   Example with multiple arguments:
#     /compare-models model_sgd model_adam
#   In the .md file:  "Compare $0 (first) and $1 (second) side by side."